# 🇹🇷 Istanbul Data Jobs Scraper + Skills Analyzer
### Data Science | Data Analyst | Machine Learning

This notebook:
1. Scrapes job postings from **LinkedIn** & **Kariyer.net**
2. Fetches full job descriptions
3. **Extracts and ranks skills** — programming languages, ML models, tools, cloud platforms, soft skills
4. Visualizes skill demand with charts
5. Exports everything to CSV

## 1. Install & Import Libraries

In [ ]:
!pip install requests beautifulsoup4 pandas lxml matplotlib seaborn wordcloud -q
print('✅ Done!')

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time, random, re, collections
from datetime import datetime
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Try importing wordcloud (optional)
try:
    from wordcloud import WordCloud
    HAS_WORDCLOUD = True
except:
    HAS_WORDCLOUD = False

print('✅ Libraries imported!')

## 2. Configuration

In [ ]:
SEARCH_KEYWORDS = [
    "data scientist",
    "data analyst",
    "machine learning engineer",
    "ML engineer",
    "veri bilimci",
    "veri analisti",
]

LOCATION      = "Istanbul"
MAX_PAGES     = 3       # pages per keyword per source
DELAY_MIN     = 2
DELAY_MAX     = 5
FETCH_DESCRIPTIONS = True  # ← set False to skip (faster but no skill extraction)

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9,tr;q=0.8",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}

print(f'✅ Config ready | Keywords: {len(SEARCH_KEYWORDS)} | Location: {LOCATION}')

## 3. Skills Dictionary

> This is the core of the analysis. Edit/extend any category to suit your needs.

In [ ]:
SKILLS = {

    # ── Programming Languages ──────────────────────────────────────────────────
    'Programming Languages': [
        'python', 'r', 'sql', 'scala', 'java', 'julia', 'c\+\+', 'c#',
        'javascript', 'typescript', 'bash', 'go', 'rust',
    ],

    # ── ML / DL Frameworks ────────────────────────────────────────────────────
    'ML / DL Frameworks': [
        'tensorflow', 'keras', 'pytorch', 'scikit-learn', 'sklearn',
        'xgboost', 'lightgbm', 'catboost', 'hugging face', 'transformers',
        'jax', 'mxnet', 'fastai', 'onnx', 'triton',
    ],

    # ── ML Models & Algorithms ────────────────────────────────────────────────
    'ML Models & Algorithms': [
        'random forest', 'gradient boosting', 'decision tree',
        'logistic regression', 'linear regression', 'svm', 'support vector',
        'k-means', 'kmeans', 'dbscan', 'pca', 'neural network',
        'convolutional neural', 'cnn', 'lstm', 'rnn', 'gru',
        'transformer', 'bert', 'gpt', 'llm', 'large language model',
        'reinforcement learning', 'time series', 'arima', 'prophet',
        'anomaly detection', 'recommendation system', 'nlp',
        'natural language processing', 'computer vision', 'object detection',
        'yolo', 'generative ai', 'diffusion model', 'rag',
        'retrieval augmented',
    ],

    # ── Data & Analytics Tools ────────────────────────────────────────────────
    'Data & Analytics Tools': [
        'pandas', 'numpy', 'scipy', 'matplotlib', 'seaborn', 'plotly',
        'tableau', 'power bi', 'looker', 'metabase', 'superset',
        'excel', 'google sheets', 'dbt', 'airflow', 'luigi',
        'jupyter', 'databricks', 'streamlit', 'gradio',
    ],

    # ── Databases & Data Warehouses ───────────────────────────────────────────
    'Databases & Warehouses': [
        'postgresql', 'mysql', 'mssql', 'sql server', 'oracle',
        'mongodb', 'cassandra', 'redis', 'elasticsearch',
        'snowflake', 'bigquery', 'redshift', 'hive', 'presto',
        'trino', 'clickhouse', 'neo4j',
    ],

    # ── Big Data & Streaming ──────────────────────────────────────────────────
    'Big Data & Streaming': [
        'spark', 'pyspark', 'hadoop', 'kafka', 'flink',
        'hbase', 'hdfs', 'yarn',
    ],

    # ── Cloud Platforms ───────────────────────────────────────────────────────
    'Cloud Platforms': [
        'aws', 'amazon web services', 'azure', 'google cloud', 'gcp',
        'sagemaker', 'vertex ai', 'azure ml', 'lambda', 'ec2', 's3',
    ],

    # ── MLOps & DevOps ────────────────────────────────────────────────────────
    'MLOps & DevOps': [
        'mlflow', 'kubeflow', 'dvc', 'bentoml', 'seldon',
        'docker', 'kubernetes', 'git', 'github', 'gitlab', 'ci/cd',
        'terraform', 'jenkins', 'fastapi', 'flask', 'rest api',
    ],

    # ── Statistics & Math ─────────────────────────────────────────────────────
    'Statistics & Math': [
        'statistics', 'probability', 'hypothesis testing', 'a/b test',
        'bayesian', 'regression analysis', 'causal inference',
        'feature engineering', 'feature selection', 'cross-validation',
        'linear algebra', 'calculus', 'optimization',
    ],

    # ── Soft Skills ───────────────────────────────────────────────────────────
    'Soft Skills': [
        'communication', 'teamwork', 'problem solving', 'analytical',
        'critical thinking', 'presentation', 'stakeholder',
        'agile', 'scrum', 'project management', 'leadership',
        'collaboration', 'storytelling', 'business acumen',
    ],
}

# Build a flat lookup: pattern → (category, display_name)
SKILL_LOOKUP = {}
for category, skills in SKILLS.items():
    for s in skills:
        display = s.replace('\\+\\+', '++').replace('\\', '')
        SKILL_LOOKUP[s] = (category, display)

print(f'✅ Skills dictionary ready: {sum(len(v) for v in SKILLS.values())} skills across {len(SKILLS)} categories')

## 4. Scraper Functions

In [ ]:
def scrape_linkedin(keyword, location="Istanbul", max_pages=3):
    jobs = []
    base_url = "https://www.linkedin.com/jobs/search"
    for page in range(max_pages):
        params = {"keywords": keyword, "location": location,
                  "start": page * 25, "position": 1, "pageNum": 0}
        try:
            r = requests.get(base_url, params=params, headers=HEADERS, timeout=15)
            if r.status_code != 200: break
            soup = BeautifulSoup(r.text, 'lxml')
            cards = soup.find_all('div', class_=re.compile('base-card'))
            if not cards:
                cards = soup.find_all('li', class_=re.compile('result-card'))
            if not cards: break
            for card in cards:
                try:
                    t = card.find(['h3','h4'], class_=re.compile('title|job-title', re.I))
                    c = card.find(['h4','a'],  class_=re.compile('company|subtitle', re.I))
                    l = card.find('span',      class_=re.compile('location', re.I))
                    a = card.find('a', href=True)
                    d = card.find('time')
                    title = t.get_text(strip=True) if t else 'N/A'
                    if title == 'N/A': continue
                    jobs.append({'source':'LinkedIn','title':title,
                        'company': c.get_text(strip=True) if c else 'N/A',
                        'location': l.get_text(strip=True) if l else location,
                        'posted_date': d.get('datetime','N/A') if d else 'N/A',
                        'link': a['href'].split('?')[0] if a else 'N/A',
                        'keyword_used': keyword, 'description': ''})
                except: continue
            print(f'  ✅ LinkedIn p{page+1}: {len(cards)} cards')
            time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))
        except Exception as e:
            print(f'  ❌ LinkedIn p{page+1}: {e}'); break
    return jobs


def scrape_kariyer(keyword, location="istanbul", max_pages=3):
    jobs = []
    base_url = "https://www.kariyer.net/is-ilanlari"
    slug = keyword.lower().replace(' ', '-')
    for page in range(1, max_pages + 1):
        try:
            r = requests.get(f"{base_url}/{slug}", params={'loc': location, 'p': page},
                             headers=HEADERS, timeout=15)
            if r.status_code != 200: break
            soup = BeautifulSoup(r.text, 'lxml')
            cards = soup.find_all('div', class_=re.compile('list-item|job-item|k-ilanList', re.I))
            if not cards:
                cards = soup.find_all('li', class_=re.compile('job|ilan', re.I))
            if not cards: break
            for card in cards:
                try:
                    t = card.find(['h2','h3','a'], class_=re.compile('title|position|jobtitle', re.I))
                    c = card.find(['span','a','p'], class_=re.compile('company|firm', re.I))
                    l = card.find(['span','div'],   class_=re.compile('location|city|sehir', re.I))
                    a = card.find('a', href=True)
                    d = card.find(['span','time'],  class_=re.compile('date|tarih', re.I))
                    title = t.get_text(strip=True) if t else 'N/A'
                    if title == 'N/A': continue
                    link = a['href'] if a else 'N/A'
                    if link != 'N/A' and not link.startswith('http'):
                        link = 'https://www.kariyer.net' + link
                    jobs.append({'source':'Kariyer.net','title':title,
                        'company': c.get_text(strip=True) if c else 'N/A',
                        'location': l.get_text(strip=True) if l else location,
                        'posted_date': d.get_text(strip=True) if d else 'N/A',
                        'link': link, 'keyword_used': keyword, 'description': ''})
                except: continue
            print(f'  ✅ Kariyer.net p{page}: {len(cards)} cards')
            time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))
        except Exception as e:
            print(f'  ❌ Kariyer.net p{page}: {e}'); break
    return jobs


def fetch_description(url, source):
    if not url or url == 'N/A': return ''
    try:
        r = requests.get(url, headers=HEADERS, timeout=15)
        if r.status_code != 200: return ''
        soup = BeautifulSoup(r.text, 'lxml')
        if source == 'LinkedIn':
            el = soup.find('div', class_=re.compile('description|details', re.I))
            if not el: el = soup.find('section', class_=re.compile('description', re.I))
        else:
            el = soup.find('div', class_=re.compile('job-description|ilan-detay|content', re.I))
        return el.get_text(' ', strip=True)[:4000] if el else ''
    except: return ''

print('✅ Scraper functions defined!')

## 5. Run Scraper

In [ ]:
all_jobs = []
print('🚀 Starting scrape...\n' + '='*60)

for kw in SEARCH_KEYWORDS:
    print(f'\n🔍 "{kw}"')
    li = scrape_linkedin(kw, LOCATION, MAX_PAGES)
    all_jobs.extend(li)
    time.sleep(random.uniform(2, 4))
    kn = scrape_kariyer(kw, 'istanbul', MAX_PAGES)
    all_jobs.extend(kn)

print(f'\n✅ Total raw: {len(all_jobs)}')

## 6. Clean & Deduplicate

In [ ]:
df = pd.DataFrame(all_jobs)
df = df[df['title'].notna() & (df['title'] != 'N/A')]
df = df.drop_duplicates(subset=['title', 'company'], keep='first')
for col in ['title','company','location']:
    df[col] = df[col].str.strip().str.replace(r'\s+', ' ', regex=True)
df['scraped_at'] = datetime.now().strftime('%Y-%m-%d %H:%M')
df = df.reset_index(drop=True)

print(f'✅ {len(df)} unique jobs after dedup')
print(df['source'].value_counts())

## 7. Fetch Full Job Descriptions
> ⚠️ Takes ~2–3 min for 100 jobs. Set `FETCH_DESCRIPTIONS = False` in config to skip.

In [ ]:
if FETCH_DESCRIPTIONS:
    print(f'Fetching descriptions for {len(df)} jobs...')
    descs = []
    for i, row in df.iterrows():
        if i % 20 == 0: print(f'  {i}/{len(df)}')
        descs.append(fetch_description(row['link'], row['source']))
        time.sleep(random.uniform(1, 2.5))
    df['description'] = descs
    filled = df['description'].str.len().gt(50).sum()
    print(f'✅ {filled}/{len(df)} descriptions successfully fetched')
else:
    print('ℹ️ Skipped — skills analysis will use job titles only')
    df['description'] = ''

## 8. 🔬 Skills Extraction
Scans each job's title + description and tags every matching skill.

In [ ]:
def extract_skills(text):
    """Return list of (category, skill_display_name) tuples found in text."""
    text = text.lower()
    found = []
    for pattern, (category, display) in SKILL_LOOKUP.items():
        # word-boundary match
        if re.search(r'(?<![a-z])' + pattern + r'(?![a-z])', text):
            found.append((category, display))
    return found

# Run extraction
df['text_combined'] = (df['title'].fillna('') + ' ' + df['description'].fillna(''))
df['skills_found']  = df['text_combined'].apply(extract_skills)

# Explode into a long-format table for analysis
rows = []
for _, row in df.iterrows():
    for (cat, skill) in row['skills_found']:
        rows.append({'job_id': row.name, 'category': cat, 'skill': skill,
                     'job_title': row['title'], 'company': row['company'],
                     'source': row['source']})
skills_df = pd.DataFrame(rows)
skills_df = skills_df.drop_duplicates(subset=['job_id','skill'])  # 1 count per job

print(f'✅ Skills extracted!')
print(f'   Total skill mentions: {len(skills_df)}')
print(f'   Unique skills found:  {skills_df["skill"].nunique()}')
print(f'   Categories covered:   {skills_df["category"].nunique()}')

## 9. 📊 Skills Analysis & Visualization

In [ ]:
# ── Helper ────────────────────────────────────────────────────────────────────
def plot_top_skills(category, n=15, color='steelblue'):
    subset = skills_df[skills_df['category'] == category]
    counts = subset['skill'].value_counts().head(n)
    if counts.empty:
        print(f'No data for: {category}'); return
    total_jobs = len(df)
    pct = (counts / total_jobs * 100).round(1)
    fig, ax = plt.subplots(figsize=(10, max(4, len(counts)*0.45)))
    bars = ax.barh(counts.index[::-1], counts.values[::-1], color=color, edgecolor='white')
    for bar, p in zip(bars, pct.values[::-1]):
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                f'{p}%', va='center', fontsize=9, color='#444')
    ax.set_xlabel('Number of Job Postings', fontsize=11)
    ax.set_title(f'🔹 {category}  (top {n})', fontsize=13, fontweight='bold', pad=12)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    sns.despine()
    plt.tight_layout()
    plt.show()

print('✅ Plot helper ready!')

In [ ]:
# ── Category overview ─────────────────────────────────────────────────────────
cat_counts = skills_df.groupby('category')['skill'].count().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(cat_counts.index, cat_counts.values,
              color=sns.color_palette('tab10', len(cat_counts)))
ax.set_title('📦 Skill Mentions by Category', fontsize=14, fontweight='bold')
ax.set_ylabel('Total Mentions')
plt.xticks(rotation=30, ha='right', fontsize=9)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(int(bar.get_height())), ha='center', fontsize=9)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# ── Programming Languages ─────────────────────────────────────────────────────
plot_top_skills('Programming Languages', color='#4C72B0')

In [ ]:
# ── ML Models & Algorithms ────────────────────────────────────────────────────
plot_top_skills('ML Models & Algorithms', color='#DD8452')

In [ ]:
# ── ML / DL Frameworks ───────────────────────────────────────────────────────
plot_top_skills('ML / DL Frameworks', color='#55A868')

In [ ]:
# ── Cloud Platforms ───────────────────────────────────────────────────────────
plot_top_skills('Cloud Platforms', color='#C44E52')

In [ ]:
# ── Databases & Warehouses ────────────────────────────────────────────────────
plot_top_skills('Databases & Warehouses', color='#8172B2')

In [ ]:
# ── Data & Analytics Tools ────────────────────────────────────────────────────
plot_top_skills('Data & Analytics Tools', color='#937860')

In [ ]:
# ── MLOps & DevOps ────────────────────────────────────────────────────────────
plot_top_skills('MLOps & DevOps', color='#DA8BC3')

In [ ]:
# ── Big Data & Streaming ──────────────────────────────────────────────────────
plot_top_skills('Big Data & Streaming', color='#8C8C8C')

In [ ]:
# ── Statistics & Math ─────────────────────────────────────────────────────────
plot_top_skills('Statistics & Math', color='#CCB974')

In [ ]:
# ── Soft Skills ───────────────────────────────────────────────────────────────
plot_top_skills('Soft Skills', color='#64B5CD')

In [ ]:
# ── Overall Top 30 Skills (all categories) ────────────────────────────────────
top30 = skills_df.groupby(['skill','category'])['job_id'].count().reset_index()
top30.columns = ['skill','category','count']
top30 = top30.sort_values('count', ascending=False).head(30)

palette = dict(zip(skills_df['category'].unique(),
                   sns.color_palette('tab10', skills_df['category'].nunique())))
colors = [palette[c] for c in top30['category']]

fig, ax = plt.subplots(figsize=(12, 10))
bars = ax.barh(top30['skill'][::-1], top30['count'][::-1], color=colors[::-1])
for bar in bars:
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
            str(int(bar.get_width())), va='center', fontsize=9)

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=palette[c], label=c) for c in palette]
ax.legend(handles=legend_elements, loc='lower right', fontsize=8)

ax.set_xlabel('Number of Job Postings')
ax.set_title('🏆 Top 30 Most In-Demand Skills — Istanbul Data Jobs', fontsize=14, fontweight='bold')
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# ── Word Cloud of all skills ──────────────────────────────────────────────────
if HAS_WORDCLOUD:
    freq = skills_df['skill'].value_counts().to_dict()
    wc = WordCloud(width=1200, height=600, background_color='white',
                   colormap='tab10', max_words=80).generate_from_frequencies(freq)
    plt.figure(figsize=(14, 7))
    plt.imshow(wc, interpolation='bilinear')
    plt.axis('off')
    plt.title('☁️ Skills Word Cloud — Istanbul Data Jobs', fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('ℹ️ wordcloud not installed — run: !pip install wordcloud')

## 10. Skills Breakdown by Job Role

In [ ]:
# Classify each job into a role bucket
def classify_role(title):
    t = title.lower()
    if any(x in t for x in ['machine learning','ml engineer','mlops','deep learning']):
        return 'ML Engineer'
    if any(x in t for x in ['data scientist','veri bilimci','bilim']):
        return 'Data Scientist'
    if any(x in t for x in ['data analyst','veri analisti','business intelligence','bi analyst']):
        return 'Data Analyst'
    if any(x in t for x in ['data engineer','veri mühendisi']):
        return 'Data Engineer'
    return 'Other'

df['role'] = df['title'].apply(classify_role)
print('Role distribution:')
print(df['role'].value_counts())

In [ ]:
# Top 10 skills per role
skills_with_role = skills_df.merge(df[['role']], left_on='job_id', right_index=True)

roles_to_plot = [r for r in ['Data Scientist','Data Analyst','ML Engineer','Data Engineer']
                 if r in skills_with_role['role'].values]

for role in roles_to_plot:
    subset = skills_with_role[skills_with_role['role'] == role]
    counts = subset['skill'].value_counts().head(12)
    if counts.empty: continue
    
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh(counts.index[::-1], counts.values[::-1], color='#4C72B0', edgecolor='white')
    ax.set_title(f'📌 Top Skills for: {role}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Job Count')
    sns.despine()
    plt.tight_layout()
    plt.show()

## 11. Skills Heatmap — Role vs Skill

In [ ]:
# Pick top 20 skills overall
top_skills = skills_df['skill'].value_counts().head(20).index.tolist()
roles      = ['Data Scientist','Data Analyst','ML Engineer','Data Engineer']

matrix = pd.DataFrame(index=roles, columns=top_skills, dtype=float).fillna(0)
for role in roles:
    role_jobs  = df[df['role'] == role].index.tolist()
    role_skills = skills_df[skills_df['job_id'].isin(role_jobs)]
    n_jobs = len(role_jobs) or 1
    for skill in top_skills:
        cnt = role_skills[role_skills['skill'] == skill]['job_id'].nunique()
        matrix.loc[role, skill] = round(cnt / n_jobs * 100, 1)

# Remove columns with all zeros
matrix = matrix.loc[:, (matrix != 0).any(axis=0)]

fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(matrix.astype(float), annot=True, fmt='.0f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, cbar_kws={'label': '% of jobs in role'})
ax.set_title('🔥 Skill Prevalence (%) by Role — Top 20 Skills', fontsize=13, fontweight='bold')
ax.set_ylabel('')
plt.xticks(rotation=40, ha='right', fontsize=9)
plt.tight_layout()
plt.show()

## 12. Summary Table

In [ ]:
summary = skills_df.groupby(['category','skill'])['job_id'].count().reset_index()
summary.columns = ['Category','Skill','Job Count']
summary['% of All Jobs'] = (summary['Job Count'] / len(df) * 100).round(1)
summary = summary.sort_values(['Category','Job Count'], ascending=[True, False])

pd.set_option('display.max_rows', 100)
summary

## 13. Export to CSV

In [ ]:
ts = datetime.now().strftime('%Y%m%d_%H%M')

# Jobs CSV
jobs_file = f'istanbul_jobs_{ts}.csv'
df.drop(columns=['text_combined','skills_found'], errors='ignore').to_csv(
    jobs_file, index=False, encoding='utf-8-sig')
print(f'✅ Jobs saved: {jobs_file}  ({len(df)} rows)')

# Skills summary CSV
skills_file = f'istanbul_skills_summary_{ts}.csv'
summary.to_csv(skills_file, index=False, encoding='utf-8-sig')
print(f'✅ Skills saved: {skills_file}  ({len(summary)} rows)')

# Download in Colab
try:
    from google.colab import files
    files.download(jobs_file)
    files.download(skills_file)
    print('⬇️ Downloads triggered!')
except:
    print('ℹ️ Files saved locally.')